# Lezione 6: Sequenze e reti neurali ricorrenti

## Obiettivi di apprendimento

Al termine di questa lezione, tu:

- Spieghi perché un perceptrone multistrato (MLP) non cattura bene i dati sequenziali
- Implementi a mano il passo ricorrente di una semplice rete neurale ricorrente (RNN), con un ciclo in PyTorch
- Individui il problema del gradiente che svanisce nelle sequenze lunghe
- Addestri una rete LSTM (memoria a lungo e breve termine, dall'inglese *Long Short-Term Memory*) su una serie temporale sintetica
- Confronti graficamente la previsione del modello con i valori reali della serie

## Dati che dipendono dall'ordine

Un perceptrone multistrato (MLP) si aspetta un vettore di dimensione fissa. Se lo addestri con frasi di cinque parole, non sa che cosa farsene di una frase di otto parole.

Inoltre, un MLP di solito concatena o somma i suoi input senza distinguere le posizioni. Per questo non distingue "il gatto insegue il cane" da "il cane insegue il gatto": stesse parole, significato opposto, ordine diverso.

Ti serve un'architettura che:

- Accetti sequenze di lunghezza variabile
- Rispetti l'ordine degli elementi
- Ricordi le informazioni dei passi precedenti mentre elabora il passo attuale

Questa architettura è la rete neurale ricorrente (RNN).

## Lo stato nascosto di una RNN

Pensa a come leggi questa stessa frase, parola per parola. Non interpreti ogni parola in modo isolato: tieni in mente un riassunto di ciò che hai letto fino a quel punto, e questo riassunto condiziona il modo in cui interpreti la parola nuova.

Una RNN fa qualcosa di simile. Mantiene un vettore di stato nascosto, hₜ, che aggiorna a ogni passo temporale combinando l'input corrente con lo stato del passo precedente:

**hₜ = tanh(Wₕ · hₜ₋₁ + Wₓ · xₜ + bₕ)**

Wₕ e Wₓ sono matrici di pesi condivise da tutti i passi temporali: la stessa regola di aggiornamento si applica al passo tre come al passo 300. Questo permette di elaborare sequenze di qualsiasi lunghezza con un numero fisso di parametri.

Implementa questo passo a mano, con un ciclo in PyTorch, per vedere come evolve lo stato nascosto man mano che entra ogni nuovo elemento della sequenza.

In [ ]:
import torch

torch.manual_seed(42)

tam_entrada = 4   # dimension de cada elemento de la secuencia
tam_oculto = 3    # dimension del estado oculto

# Pesos aleatorios, como los tendria una capa RNN recien creada, sin entrenar
Wx = torch.randn(tam_oculto, tam_entrada) * 0.5
Wh = torch.randn(tam_oculto, tam_oculto) * 0.5
bh = torch.zeros(tam_oculto)

# Una secuencia de ejemplo: seis pasos temporales, cuatro valores por paso
secuencia = torch.randn(6, tam_entrada)

h = torch.zeros(tam_oculto)  # estado oculto inicial h_0, en cero

print("Evolucion del estado oculto paso a paso:\n")
for t, x_t in enumerate(secuencia):
    # el mismo calculo, con los mismos pesos Wx y Wh, se repite en cada paso temporal
    h = torch.tanh(Wh @ h + Wx @ x_t + bh)
    print(f"Paso {t + 1}: h = {h.numpy().round(3)}")

## Il gradiente che svanisce

Semplifica al massimo la ricorrenza: ignora gli input e i bias, e supponi che Wₕ si riduca a un unico numero w. L'aggiornamento hₜ = w · hₜ₋₁ si srotola così:

**hₜ = wᵗ · h₀**

Se il valore assoluto di w supera uno, wᵗ cresce senza controllo man mano che t avanza: è il gradiente esplosivo. Se il valore assoluto di w è minore di uno, wᵗ decade verso zero: è il gradiente che svanisce.

In pratica, l'aggiornamento passa inoltre attraverso una tangente iperbolica, la cui derivata non supera mai uno. Il gradiente che viene retropropagato attraverso molti passi temporali è, in sostanza, un prodotto di molti fattori come questo. Basta che ogni fattore sia leggermente minore di uno perché, dopo decine o centinaia di passi, il prodotto si avvicini a zero.

La conseguenza è grave: se il gradiente che arriva a un passo lontano è praticamente nullo, quella parte della rete smette di ricevere segnale di apprendimento, e il modello non impara le dipendenze a distanza, anche se avrebbe la capacità teorica di rappresentarle.

## LSTM e GRU in soccorso

La rete LSTM (memoria a lungo e breve termine) separa due flussi di informazione: uno stato di cella, cₜ, che funge da memoria a lungo termine con aggiornamenti per lo più additivi, e uno stato nascosto hₜ, calcolato a partire dallo stato di cella a ogni passo.

Tre porte, ciascuna con attivazione sigmoide, regolano l'accesso a questa memoria:

- **Porta di dimenticanza**: decide quale parte dello stato di cella precedente conservare
- **Porta di ingresso**: decide quale informazione nuova incorporare
- **Porta di uscita**: decide quale parte dello stato di cella esporre come stato nascosto

L'aggiornamento dello stato di cella, cₜ = fₜ ⊙ cₜ₋₁ + iₜ ⊙ c̃ₜ, è in gran parte una somma, non una riscrittura completa. Questo carattere additivo è la chiave: quando la porta di dimenticanza è vicina a uno, il gradiente fluisce all'indietro quasi senza attenuarsi.

La GRU (unità ricorrente con porte, dall'inglese *Gated Recurrent Unit*) semplifica la LSTM: fonde lo stato di cella e quello nascosto in un unico vettore e riduce le porte a due (aggiornamento e reset). Ha meno parametri, si addestra un po' più in fretta e, in molti compiti, dà prestazioni comparabili.

Addestra ora una piccola LSTM con `torch.nn.LSTM` su una serie temporale sintetica: un seno con rumore. Come funzione di perdita usi l'errore quadratico medio (MSE, dall'inglese *Mean Squared Error*), tipico dei compiti di regressione.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# Serie temporal sintetica: un seno con ruido gaussiano
pasos_totales = 500
t = np.linspace(0, 50, pasos_totales)
serie = (np.sin(t) + np.random.normal(0, 0.1, size=pasos_totales)).astype(np.float32)

plt.figure(figsize=(10, 3))
plt.plot(serie)
plt.title("Serie temporal sintetica: seno con ruido")
plt.xlabel("Paso temporal")
plt.ylabel("Valor")
plt.show()

longitud_ventana = 20  # cuantos pasos pasados usa el modelo para predecir el siguiente

def crear_ventanas(serie, longitud_ventana):
    X, y = [], []
    for i in range(len(serie) - longitud_ventana):
        X.append(serie[i:i + longitud_ventana])
        y.append(serie[i + longitud_ventana])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = crear_ventanas(serie, longitud_ventana)

# Reservamos una parte para entrenar y el resto para evaluar, respetando el orden temporal
corte = int(len(X) * 0.8)
X_entreno, X_prueba = X[:corte], X[corte:]
y_entreno, y_prueba = y[:corte], y[corte:]

# nn.LSTM espera tensores con forma (lote, tiempo, caracteristicas)
X_entreno_t = torch.tensor(X_entreno).unsqueeze(-1)
y_entreno_t = torch.tensor(y_entreno).unsqueeze(-1)
X_prueba_t = torch.tensor(X_prueba).unsqueeze(-1)
y_prueba_t = torch.tensor(y_prueba).unsqueeze(-1)

print("Forma de X_entreno:", X_entreno_t.shape)

In [ ]:
class ModeloLSTM(nn.Module):
    def __init__(self, tam_oculto=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=tam_oculto, batch_first=True)
        self.salida = nn.Linear(tam_oculto, 1)

    def forward(self, x):
        salida_lstm, _ = self.lstm(x)
        ultimo_paso = salida_lstm[:, -1, :]  # el ultimo estado oculto resume toda la ventana
        return self.salida(ultimo_paso)

modelo = ModeloLSTM()
perdida_fn = nn.MSELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)

num_epocas = 150
historial_perdida = []

for epoca in range(num_epocas):
    modelo.train()
    prediccion = modelo(X_entreno_t)
    perdida = perdida_fn(prediccion, y_entreno_t)
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()
    historial_perdida.append(perdida.item())
    if (epoca + 1) % 30 == 0:
        print(f"Epoca {epoca + 1}/{num_epocas} - perdida: {perdida.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(historial_perdida)
plt.xlabel("Epoca")
plt.ylabel("Perdida (MSE)")
plt.title("Curva de perdida durante el entrenamiento")
plt.show()

In [ ]:
modelo.eval()
with torch.no_grad():
    prediccion_prueba = modelo(X_prueba_t).squeeze().numpy()

plt.figure(figsize=(10, 4))
plt.plot(y_prueba, label="Valor real")
plt.plot(prediccion_prueba, label="Prediccion de la LSTM")
plt.xlabel("Paso temporal (conjunto de prueba)")
plt.ylabel("Valor")
plt.title("Prediccion frente a valores reales")
plt.legend()
plt.show()

## Riepilogo dei punti chiave

- Una sequenza è una collezione ordinata di elementi di lunghezza variabile: il testo, l'audio e le serie temporali ne sono esempi tipici
- Un MLP richiede input di dimensione fissa e non cattura l'ordine; una RNN mantiene uno stato nascosto che riassume quanto visto finora
- Lo stato nascosto si aggiorna con hₜ = tanh(Wₕ · hₜ₋₁ + Wₓ · xₜ + bₕ), con gli stessi pesi condivisi in tutti i passi temporali
- Nelle sequenze lunghe, la RNN semplice soffre del problema del gradiente che svanisce o esplode, e smette di apprendere le dipendenze lontane
- La LSTM introduce uno stato di cella con aggiornamento additivo e tre porte (dimenticanza, ingresso, uscita) che proteggono la memoria a lungo termine
- La GRU semplifica la LSTM con due porte (aggiornamento e reset) e meno parametri
- Entrambe le architetture attenuano lo svanimento del gradiente perché lasciano fluire l'informazione con modifiche minime attraverso molti passi

**Apri ora il quaderno di pratica, `../notebooks_practica/clase06_practica_secuencias_redes_recurrentes.ipynb`, e metti in funzione ciò che hai appena visto.**